# Build a mixed-file data analysis agent

Live stream companion. We first make the tools work, then compare LangGraph and LlamaIndex with the same tool set.

Install from this repository with `pip install -e '.[notebook]'`. For agent cells, also install `.[langgraph]` or `.[llamaindex]` and set `OPENAI_API_KEY`.

## 1. Ingest the demo files

CSV/XLSX become DuckDB tables. PDF/DOCX and text files become source-located passages. Swap `data/sample` with your own folder when ready.

In [ ]:
from pathlib import Path
from data_agent.ingest import ingest_folder, list_tables, run_readonly_sql, search_documents

DATA = Path('../data/sample').resolve()
DB = Path('../.data-agent/live-demo.duckdb').resolve()
summary = ingest_folder(DATA, DB)
summary

In [ ]:
list_tables(DB)

## 2. Ask the table a precise question

This query does not ask an LLM to calculate. We use a read-only query over ingested table names and return rows for the final answer.

In [ ]:
run_readonly_sql(DB, '''
SELECT region, SUM(revenue) AS revenue
FROM sales
GROUP BY region
ORDER BY revenue DESC
''')

## 3. Find narrative evidence

The starter uses lexical overlap to keep this demo local and transparent. It is a baseline, not a claim that lexical retrieval is best. Add labeled queries and compare recall@k before switching to vector or hybrid search.

In [ ]:
search_documents(DB, 'What explains North Widget demand and what may happen next month?')

## 4. Compare framework orchestration

Install one optional adapter at a time. Keep the model, prompt, tools, and test cases fixed.

```bash
pip install -e '.[langgraph]'
# or
pip install -e '.[llamaindex]'
```

LangGraph uses an explicit graph and tool loop. LlamaIndex uses `AgentWorkflow`. Both call the same `catalog`, `query`, and `find_evidence` functions. Track traces and score outputs with `evals/cases.json`.

In [ ]:
# Optional: run LangGraph after installing its extra and setting OPENAI_API_KEY.
# from data_agent.frameworks.langgraph_app import build
# app = build(str(DB))
# result = app.invoke({'messages': [{'role': 'user', 'content':
#     'How much revenue did North make, and what explains Widget demand there?'}]})
# result['messages'][-1].content

In [ ]:
# Optional: run LlamaIndex in an async cell after installing its extra.
# from data_agent.frameworks.llamaindex_app import build
# workflow = build(str(DB))
# response = await workflow.run(user_msg=
#     'How much revenue did North make, and what explains Widget demand there?')
# response

## 5. Finish with evals and a performance baseline

Use the fixed cases in `evals/cases.json`. Add a few representative questions for every input shape and an unanswerable case. Compare exact SQL rows deterministically, verify citation coverage, and use an LLM judge only for qualitative grounding checks.

Record parse throughput by file type, end-to-end p50/p95 latency, tokens/cost, memory, tool-call count, and failure rate. For a corpus of hundreds of files, add hash-based incremental ingestion, a durable queue, bounded workers, retries, and access-control-aware indexes. The starter intentionally does not claim production-scale throughput.